In [1]:
#@markdown It will restart the kernel (session), don't worry. #CAUTION USE ABSOLUTE FILE PATHS like kaggle/working/md.gro ,kaggle/working/md.xtc otherwise generated file will be invisible.
!pip install -q condacolab
import condacolab
condacolab.install()

⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:09
🔁 Restarting kernel...


In [1]:
!conda install ambertools


Channels:
 - conda-forge
Platform: linux-64
Solving environment: done

## Package Plan ##

  environment location: /usr/local

  added / updated specs:
    - ambertools


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _openmp_mutex-4.5          |       7_kmp_llvm           8 KB  conda-forge
    ambertools-24.8            |cuda_None_nompi_py311h9353165_101        96.5 MB  conda-forge
    arpack-3.9.1               |nompi_hf03ea27_102         127 KB  conda-forge
    blosc-1.21.6               |       he440d0b_1          47 KB  conda-forge
    brotli-1.1.0               |       hb9d3cd8_2          19 KB  conda-forge
    brotli-bin-1.1.0           |       hb9d3cd8_2          18 KB  conda-forge
    ca-certificates-2026.6.17  |       hbd8a1cb_0         126 KB  conda-forge
    certifi-2026.6.17          |     pyhd8ed1ab_0         131 KB  conda-forge
    conda-24.11.3              |  py311h38be

In [2]:
%%bash
set -e

# 0. Verify GPU
nvidia-smi

# 1. Install build deps + CUDA headers
apt-get update -qq
apt-get install -y -qq \
    build-essential cmake git \
    fftw3-dev libfftw3-dev \
    libgsl-dev libhwloc-dev \
    nvidia-cuda-dev

# 2. Download & unpack GROMACS 2024.2
wget -q https://ftp.gromacs.org/pub/gromacs/gromacs-2024.2.tar.gz
tar xfz gromacs-2024.2.tar.gz

# 3. Create & enter build directory
mkdir -p gromacs-2024.2/build
cd gromacs-2024.2/build

# 4. Configure with CUDA, bundled FFTW & tests
cmake .. \
    -DGMX_BUILD_OWN_FFTW=ON \
    -DREGRESSIONTEST_DOWNLOAD=ON \
    -DGMX_GPU=CUDA \
    -DGMX_CUDA_TARGET_SM=75 \
    -DGMX_USE_OPENCL=OFF \
    -DCMAKE_INSTALL_PREFIX=/kaggle/working/gromacs_installed

# 5. Install tqdm for progress bars
pip install --quiet tqdm

# 6. Build with tqdm progress
python3 << 'EOF'
import os, subprocess
from tqdm import tqdm

cpu = os.cpu_count() or 1
# dry-run to count steps
dry = subprocess.run(
    ["make", "-n", f"-j{cpu}"],
    stdout=subprocess.PIPE, text=True
)
steps = len(dry.stdout.splitlines())

# actual make
proc = subprocess.Popen(
    ["make", f"-j{cpu}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True
)
with tqdm(total=steps, desc="Building GROMACS") as p:
    for _ in proc.stdout:
        p.update(1)
proc.wait()
EOF

# 7. Install into the working directory
make install

# 8. Ensure executables are runnable
chmod +x /kaggle/working/gromacs_installed/bin/*

echo "✅ GROMACS installed to /kaggle/working/gromacs_installed"


Fri Jul  3 06:49:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla P100-PCIE-16GB           Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P0             26W /  250W |       0MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
CMake Deprecation Warning at src/external/googletest/CMakeLists.txt:4 (cmake_minimum_required):
  Compatibility with CMake < 3.10 will be removed from a future version of
  CMake.

  Update the VERSION argument <min> value.  Or, use the <min>...<max> syntax
  to tell CMake that the project requires at least <min> but has been updated
  to work with policies introduced by <max> or earlier.


CMake Deprecation Warning at src/external/googletest/googlemock/CMakeLists.txt:39 (cmake_minimum_required):
  Compatibility with CMake < 3.10 will be removed from a future version of
  CMake.

  Update the VERSION argument <min> value.  Or, use the <min>...<max> syntax
  to tell CMake that the project requires at least <min> but has been updated
  to work with policies introduced by <max> or earlier.


CMake Depre

In [3]:
!chmod +x /kaggle/working/gromacs_installed/bin/*
#Reuse this cell to set the gromacs execuables(use gromacs without reinstalling everytime) 


In [4]:
%%bash
cd /kaggle/working/gromacs_installed/lib

# 1. Symlink libgromacs.so.9 if you haven’t already
ln -sf libgromacs.so.9.0.0 libgromacs.so.9

# 2. Symlink libmuparser.so.2 → the actual file
ln -sf libmuparser.so.2.3.4 libmuparser.so.2

# (If the next missing .so is libnblib_gmx.so.0 or similar, do the same:
# ln -sf libnblib_gmx.so.0.1.0 libnblib_gmx.so.0 )

# 3. Source GROMACS env-setup
source /kaggle/working/gromacs_installed/bin/GMXRC

# 4. Test
gmx --version
#Test GROMACS AND RUN THIS AFTER THE PREVIOUS CELL 

                         :-) GROMACS - gmx, 2024.2 (-:

Executable:   /kaggle/working/gromacs_installed/bin/gmx
Data prefix:  /kaggle/working/gromacs_installed
Working dir:  /kaggle/working/gromacs_installed/lib
Command line:
  gmx --version

GROMACS version:     2024.2
Precision:           mixed
Memory model:        64 bit
MPI library:         thread_mpi
OpenMP support:      enabled (GMX_OPENMP_MAX_THREADS = 128)
GPU support:         CUDA
NBNxM GPU setup:     super-cluster 2x2x2 / cluster 8
SIMD instructions:   AVX_512
CPU FFT library:     fftw-3.3.8-sse2-avx-avx2-avx2_128-avx512
GPU FFT library:     cuFFT
Multi-GPU FFT:       none
RDTSCP usage:        enabled
TNG support:         enabled
Hwloc support:       disabled
Tracing support:     disabled
C compiler:          /usr/bin/cc GNU 11.4.0
C compiler flags:    -fexcess-precision=fast -funroll-all-loops -march=skylake-avx512 -Wno-missing-field-initializers -O3 -DNDEBUG
C++ compiler:        /usr/bin/c++ GNU 11.4.0
C++ compiler flags:  

In [5]:
# using pip , THIS IS FOR THE ANALYSIS PART
!pip install -qq numpy matplotlib MDAnalysis netCDF4 gmx_MMPBSA 
# (this will pull in scipy, networkx, parmed, python-dateutil, six, etc)


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 45.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 6.0 MB/s eta 0:00:0000:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 58.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 64.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 70.7 MB/s eta 0:00:00


In [6]:
pwd #current directory

'/kaggle/working'

In [7]:
!gmx_MMPBSA -h
##mmpbsa -Help


[INFO   ] Starting gmx_MMPBSA 1.6.5
[INFO   ] Command-line
  gmx_MMPBSA -h

usage: gmx_MMPBSA [-h] [-v] [--input-file-help]
                  [--create_input [{gb,pb,rism,ala,decomp,nmode,gbnsr6,all} ...]]
                  [-O] [-prefix <file prefix>] [-i FILE] [-xvvfile XVVFILE]
                  [-o FILE] [-do FILE] [-eo FILE] [-deo FILE] [-nogui] [-s]
                  [-cs <Structure File>] [-ci <Index File>] [-cg index index]
                  [-ct [TRJ ...]] [-cp <Topology>] [-cr <PDB File>]
                  [-rs <Structure File>] [-ri <Index File>] [-rg index]
                  [-rt [TRJ ...]] [-rp <Topology>] [-lm <Structure File>]
                  [-ls <Structure File>] [-li <Index File>] [-lg index]
                  [-lt [TRJ ...]] [-lp <Topology>] [--rewrite-output]
                  [--clean]

gmx_MMPBSA is a new tool based on AMBER's MMPBSA.py aiming to perform end-
state free energy calculations with GROMACS files. This program is an
adaptation of Amber's MMPBSA.py an

In [ ]:
!gmx_MMPBSA -O \
 -i /kaggle/working/gmx_MMPBSA/examples/Protein_DNA/mmpbsa.in \
 -cs /kaggle/working/gmx_MMPBSA/examples/Protein_DNA/com.tpr \
 -ct /kaggle/working/gmx_MMPBSA/examples/Protein_DNA/com_traj.xtc \
 -ci /kaggle/working/gmx_MMPBSA/examples/Protein_DNA/index.ndx \
 -cg 3 4 \
 -cp /kaggle/working/gmx_MMPBSA/examples/Protein_DNA/topol.top \
 -o /kaggle/working/gmx_MMPBSA/examples/Protein_DNA/FINAL_RESULTS_MMPBSA.dat \
 -eo /kaggle/working/gmx_MMPBSA/examples/Protein_DNA/FINAL_RESULTS_MMPBSA.csv
#mmpbsa command, just change the file paths according to your files and upload the mmpbsa.in file according to your simulations.


In [ ]:
###BELOW FOR FEL SHAM , PC1 PC2 #COVAR ETC  

In [ ]:

%%bash
export GRO=/kaggle/working/output.gro
export XTC=/kaggle/working/com_traj.xtc
export VMIN=0
export VMAX=17

echo -e "Variables set:\n  GRO   = $GRO\n  XTC   = $XTC\n  VMIN  = $VMIN\n  VMAX  = $VMAX"


In [ ]:
%%bash
set -e

GRO=/kaggle/working/output.gro
XTC=/kaggle/working/com_traj.xtc

printf "1\n1\n" | gmx covar -quiet \
  -s "$GRO" -f "$XTC" \
  -o eigenvalues.xvg \
  -v eigenvectors.trr \
  -xpma covar.xpm
echo "✅ Covariance & eigenvectors done"
ls -l eigenvalues.xvg eigenvectors.trr covar.xpm


In [ ]:
%%bash
set -e

GRO=/kaggle/working/output.gro
XTC=/kaggle/working/com_traj.xtc

# Pipe “1” into both of anaeig’s prompts (fit‐group and analysis‐group)
printf "1\n1\n" | gmx anaeig -quiet \
  -f "$XTC" \
  -s "$GRO" \
  -v eigenvectors.trr \
  -last 1 \
  -proj pc1.xvg

echo "✅ PC1 projection done"
ls -l pc1.xvg


In [ ]:
%%bash
set -e

GRO=/kaggle/working/output.gro
XTC=/kaggle/working/com_traj.xtc

# Pipe “1” into both prompts again—this covers any extra prompt before selecting PC 2
printf "1\n1\n" | gmx anaeig -quiet \
  -f "$XTC" \
  -s "$GRO" \
  -v eigenvectors.trr \
  -first 2 \
  -last 2 \
  -proj pc2.xvg

echo "✅ PC2 projection done"
ls -l pc2.xvg


In [ ]:
%%bash
set -e
paste /kaggle/working/pc1.xvg /kaggle/working/pc2.xvg \
  | awk '{print $1, $2, $4}' > /kaggle/working/PC1PC2.xvg
echo "✅ Merged PC1 & PC2 → PC1PC2.xvg"
ls -l /kaggle/working/PC1PC2.xvg
##PC1 and PC2 

In [ ]:
%%bash
set -e
gmx sham -quiet \
  -f /kaggle/working/PC1PC2.xvg \
  -ls /kaggle/working/FES.xpm
echo "✅ FES calculation done"
ls -l /kaggle/working/FES.xpm


In [ ]:
%%bash
set -e

# Create a small Python script that contains your original converter
cat > convert_xpm2dat.py << 'EOF'
#!/usr/bin/env python3
import sys
import re

def xpm2txt(xpm_file, out_file, sort_col=None):
    """
    Converts an XPM file (from gmx sham) to a 3-column text file.
    """
    with open(xpm_file, 'r') as f:
        lines = f.readlines()

    x_axis = []
    y_axis = []
    xpm_data = []
    letter_to_value = {}

    for line in lines:
        if line.startswith("/* x-axis"):
            tokens = line.split()
            x_axis = list(map(float, tokens[2:-2]))
        if line.startswith("/* y-axis"):
            tokens = line.split()
            y_axis = list(map(float, tokens[2:-2]))
        if line.startswith('"') and x_axis and y_axis:
            cleaned = line.strip().strip(',')[1:-1]
            xpm_data.insert(0, cleaned)
        if line.startswith('"') and len(line.split()) > 4:
            tokens = line.split()
            letter = tokens[0][1:]
            value  = float(tokens[-2][1:-1])
            letter_to_value[letter] = value

    with open(out_file, 'w') as fout:
        for y_index, data_value in enumerate(xpm_data):
            y_val = y_axis[y_index]
            for x_index, x_val in enumerate(x_axis):
                fout.write("{:3.5f}\t{:3.5f}\t{:3.5f}\n".format(
                    x_val, y_val, letter_to_value[data_value[x_index]]))

    print(f"Converted {xpm_file} -> {out_file}")

if __name__ == "__main__":
    if len(sys.argv) != 3:
        print("Usage: convert_xpm2dat.py input.xpm output.dat")
        sys.exit(1)
    xpm2txt(sys.argv[1], sys.argv[2])
EOF

chmod +x convert_xpm2dat.py

# Run it exactly as you had
./convert_xpm2dat.py /kaggle/working/FES.xpm /kaggle/working/fel.dat

# Show the first few lines to confirm
head -n5 /kaggle/working/fel.dat


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
#set the Vmin and Vmax from the fel.dat file
# Parameters
fel_file = "/kaggle/working/fel.dat"
vmin = 0.0
vmax = 17.0

# Load data
data = np.loadtxt(fel_file)
pc1, pc2, energy = data[:, 0], data[:, 1], data[:, 2]

# Create grid
xi = np.linspace(pc1.min(), pc1.max(), 100)
yi = np.linspace(pc2.min(), pc2.max(), 100)
xi, yi = np.meshgrid(xi, yi)
zi = griddata((pc1, pc2), energy, (xi, yi), method='linear')

# 3D Free Energy Landscape
fig3d = plt.figure(figsize=(8, 6))
ax = fig3d.add_subplot(111, projection='3d')
surface = ax.plot_surface(xi, yi, zi, cmap='jet', edgecolor='none', vmin=vmin, vmax=vmax)
ax.contourf(xi, yi, zi, zdir='z', offset=zi.min(), levels=25, cmap='jet', vmin=vmin, vmax=vmax)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
fig3d.colorbar(surface, shrink=0.5, label='kJ/mol')
fig3d.savefig("/kaggle/working/FEL3D.png", dpi=300)
plt.close(fig3d)
print("✅ Saved 3D plot as FEL3D.png")

# 2D Free Energy Contour
fig2d = plt.figure(figsize=(6, 5))
cf = plt.contourf(xi, yi, zi, levels=20, cmap='jet', vmin=vmin, vmax=vmax)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.colorbar(cf, label='kJ/mol')
fig2d.savefig("/kaggle/working/FEL2D.png", dpi=200)
plt.close(fig2d)
print("✅ Saved 2D plot as FEL2D.png")

!ls -l /kaggle/working/FEL2D.png /kaggle/working/FEL3D.png


In [ ]:
%%bash
# ─── User‐configurable inputs ─────────────────────────────────────────
TPR="/kaggle/working/com.tpr"             # .tpr or processed .gro
TRAJ="/kaggle/working/com_traj.xtc"    # trajectory
EDR="/kaggle/working/md.edr"                      # GROMACS energy file

# ─── 1) RMSD: Protein (ns) ──────────────────────────────
echo -e "Protein" | gmx rms \
  -s "$TPR" -f "$TRAJ" -o /kaggle/working/rmsd.xvg -tu ns &> /dev/null
echo "✅ RMSD done"

# ─── 2) RMSF: residue‐wise fluctuation of Protein backbone ──────────
echo -e "Backbone" | gmx rmsf \
  -s "$TPR" -f "$TRAJ" -o /kaggle/working/rmsf_residue.xvg -res &> /dev/null
echo "✅ RMSF done"

# ─── 3) Radius of gyration: Protein ─────────────────────────────────
echo -e "Protein" | gmx gyrate \
  -s "$TPR" -f "$TRAJ" -o /kaggle/working/gyrate.xvg &> /dev/null
echo "✅ Gyration done"

# ─── 4) SASA: total and residue‐wise (Protein vs. SOL) ─────────────
echo -e "Protein\nSOL" | gmx sasa \
  -s "$TPR" -f "$TRAJ" \
  -o /kaggle/working/sasa_total.xvg \
  -or /kaggle/working/sasa_residue.xvg &> /dev/null
echo "✅ SASA done"

# ─── 5) Hydrogen bonds: count H‐bonds within Protein ────────────────
echo -e "Protein\nProtein" | gmx hbond \
  -s "$TPR" -f "$TRAJ" -num /kaggle/working/hbnum.xvg &> /dev/null
echo "✅ H-bonds done"

# ─── 6) Total energy time‐series ────────────────────────────────────
echo -e "Total Energy" | gmx energy \
  -f "$EDR" -o /kaggle/working/energy_total.xvg &> /dev/null
echo "✅ Total energy done"

echo "🎯 All analysis complete. Files saved in /kaggle/working/"


In [ ]:
#DO DSSP analysis
import subprocess

# your inputs
xtc    = "/kaggle/working/com_traj.xtc"
tpr    = "/kaggle/working/com.tpr"
group1 = 1
group2 = 1

# build the command, now including -hmode dssp
cmd = [
    "gmx", "dssp",
    "-f", xtc,
    "-s", tpr,
    "-hmode", "dssp",        # ← DSSP will build its own pseudo‐hydrogens
    "-o", "/kaggle/working/dssp.dat",   # use .dat is gromacs is 2024
    "-num", "/kaggle/working/count.xvg" # use num instead of sc if gmx 2024
]

# feed the two group selections on stdin
subprocess.run(
    cmd,
    input=f"{group1}\n{group2}\n".encode(),
    check=True
)

print("Done: dssp.dat and scount.xvg written")


In [ ]:
#!/usr/bin/env python
# Calculate correlation in MD trajectory 
# Author: SRAY
# Date: 17-01-2025

import numpy as np
import math
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import cm, colors
import MDAnalysis as mda
import logging

# Configure the logger
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
log = logging.getLogger(__name__)

# Function to parse the trajectory with atom selection
def parse_traj(traj, topology, step=1, selected_atoms="name CA", lazy_load=False):
    u = mda.Universe(topology, traj)
    atom_selection = u.select_atoms(selected_atoms)
    residues = {}
    for ts in u.trajectory[::step]:
        for atom in atom_selection:
            res = atom.resid
            co_ords = atom.position.tolist()
            if res in residues:
                residues[res].append(co_ords)
            else:
                residues[res] = [co_ords]
    return residues

# Function to compute mean dot product
def mean_dot(m1, m2, size):
    DOT = np.zeros(size)
    for t in range(size):
        DOT[t] = np.dot(m1[t], m2[t])
    return np.mean(DOT)

# Function to compute correlation
def correlate(residues):
    sorted_residues = sorted(residues.keys())
    num_trajectories = len(residues[sorted_residues[0]])
    num_residues = len(residues)
    correlation = np.zeros((num_residues, num_residues))
    for a, key_a in enumerate(sorted_residues):
        i = residues[key_a]
        resI = np.array(i)
        meanI = np.tile((np.mean(resI, 0)), (num_trajectories, 1))
        idelta = resI - meanI
        magnitudeI = math.sqrt(mean_dot(idelta, idelta, num_trajectories))
        for b, key_b in enumerate(sorted_residues):
            j = residues[key_b]
            resJ = np.array(j)
            meanJ = np.tile((np.mean(resJ, 0)), (num_trajectories, 1))
            jdelta = resJ - meanJ
            magnitudeJ = math.sqrt(mean_dot(jdelta, jdelta, num_trajectories))
            meanDotIJ = mean_dot(idelta, jdelta, num_trajectories)
            magProd = magnitudeI * magnitudeJ
            correlation[a, b] = meanDotIJ / magProd
    return correlation

# Function to plot and save the heat map with a custom colormap
def plot_map(correlation, title, output_prefix):
    M = np.array(correlation)
    fig, ax = plt.subplots()

    # Create a custom colormap from deep pink to white to deep cyan
    custom_cmap = colors.LinearSegmentedColormap.from_list(
        "deep_pink_white_cyan", ["#FF1493", "white", "#008B8B"], N=300
    )

    heatmap = ax.pcolor(M, cmap=custom_cmap, vmin=-1, vmax=1)
    ax.set_frame_on(False)
    ax.grid(False)
    plt.xticks(rotation=90, fontsize=8)
    plt.yticks(fontsize=8)
    plt.title(title, fontsize=16)
    plt.xlabel('Residue Index', fontsize=12)
    plt.ylabel("Residue Index", fontsize=12)
    plt.colorbar(heatmap, orientation="vertical")
    plt.savefig(f"{output_prefix}.png", dpi=300, format="png")
    plt.close()

# Function to print correlation matrix to a file
def print_correlation(correlation, output_prefix):
    rows = correlation.shape[0]
    cols = correlation.shape[1]
    with open(f"{output_prefix}.txt", "w") as w:
        for r in range(rows):
            for c in range(cols):
                w.write(f'{correlation[r, c]} ')
            w.write('\n')

# Main function to execute
def main():
    trajectory = "/kaggle/working/com_traj.xtc"  # Replace with your trajectory file path
    topology = "/kaggle/working/output.gro"  # Replace with your topology file path
    step = 1
    lazy_load = True
    title = "Unnamed"
    prefix = "correlation"

    log.info("Preparing a trajectory matrix...")
    traj_matrix = parse_traj(trajectory, topology, step, lazy_load=lazy_load)

    log.info("Correlating...")
    correlation = correlate(traj_matrix)

    log.info("Plotting heat map...")
    plot_map(correlation, title, prefix)
    print_correlation(correlation, prefix)

# Run the main function
if __name__ == "__main__":
    main()


In [ ]:
#USING DAT FILE OF THE DSSP
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from matplotlib.colors import ListedColormap

# -----------------------------
# 1. Load & parse the DSSP file
# -----------------------------
with open("/kaggle/working/dssp.dat", "r") as file:
    lines = file.readlines()

# Skip any header lines starting with "#" or "@"
data = [list(line.strip()) for line in lines if not line.startswith("#") and not line.startswith("@")]

# -----------------------------
# 2. Define all possible symbols & their colors/labels
# -----------------------------
all_symbols = ["H", "B", "E", "G", "I", "P", "S", "T", "~"]

symbol_color_map = {
    "H": "#ff0000",   # Alpha‐helix (red)
    "B": "#ff8000",   # Isolated beta‐bridge (orange)
    "E": "#ffff00",   # Extended beta‐strand (yellow)
    "G": "#80ff00",   # 3_10‐helix (lime green)
    "I": "#00ff80",   # Pi‐helix (aqua-green)
    "P": "#00ffff",   # Polyproline helix (cyan)
    "S": "#0080ff",   # Bend (blue)
    "T": "#8000ff",   # Turn (violet)
    "~": "pink"    # Loop/Coil (magenta)
}


symbol_label_map = {
    "H": "Alpha‐helix",
    "B": "Isolated beta‐bridge",
    "E": "Extended beta‐strand",
    "G": "3₁₀‐helix",
    "I": "Pi‐helix",
    "P": "Polyproline helix",
    "S": "Bend",
    "T": "Turn",
    "~": "Loop/Coil"
}

# -----------------------------
# 3. Find which symbols actually appear in the file
# -----------------------------
present_symbols = [s for s in all_symbols if any(s in row for row in data)]

# Build a numeric index only for those present symbols
# e.g. if present_symbols = ["H","E","~"], then {"H":0, "E":1, "~":2}
symbol_to_index = {sym: idx for idx, sym in enumerate(present_symbols)}

# -----------------------------
# 4. Convert the character grid → numeric grid
# -----------------------------
numeric_data = np.array([
    [symbol_to_index.get(char, np.nan) for char in row]
    for row in data
])

# -----------------------------
# 5. Build a colormap of exactly one color per present symbol
# -----------------------------
present_colors = [symbol_color_map[sym] for sym in present_symbols]
cmap = ListedColormap(present_colors)

# -----------------------------
# 6. Plot the heatmap
# -----------------------------
fig, ax = plt.subplots(figsize=(10, 8))

# Transpose so that x‐axis = “time” (columns), y‐axis = residue (rows).
# origin="lower" makes “residue 0” appear at the bottom.
im = ax.imshow(
    numeric_data.T,
    aspect="auto",
    cmap=cmap,
    origin="lower",
    interpolation="nearest"
)

# -----------------------------
# 7. Tidy up axes & spines for a cleaner look
# -----------------------------
# Remove top/right spines
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Only draw left & bottom spines (optional; cleaner)
ax.spines["bottom"].set_color("#333333")
ax.spines["left"].set_color("#333333")

# Increase tick‐label font size
ax.tick_params(axis="x", labelsize=12)
ax.tick_params(axis="y", labelsize=12)

ax.set_xlabel("Time (frame index)", fontsize=14, color="#222222")
ax.set_ylabel("Residue Number", fontsize=14, color="#222222")
ax.set_title("Secondary Structure Heatmap Over Time (WT 600)", fontsize=16, pad=12)

# If there are many columns (timepoints), you may want to only label every Nth tick:
# Uncomment & adjust N as needed:
# N = max(1, numeric_data.shape[1] // 10)   # label ~10 ticks evenly
# ax.set_xticks(np.arange(0, numeric_data.shape[1], N))
# ax.set_xticklabels(np.arange(0, numeric_data.shape[1], N))

# Same for the y‐axis if there are many residues:
# M = max(1, numeric_data.shape[0] // 10)
# ax.set_yticks(np.arange(0, numeric_data.shape[0], M))
# ax.set_yticklabels(np.arange(0, numeric_data.shape[0], M))

# -----------------------------
# 8. Build a discrete legend (one patch per symbol)
# -----------------------------
patches = []
for sym in present_symbols:
    color = symbol_color_map[sym]
    label = f"{sym} ({symbol_label_map[sym]})"
    patches.append(mpatches.Patch(color=color, label=label))

# Place the legend outside the plot on the right
ax.legend(
    handles=patches,
    bbox_to_anchor=(1.05, 1.0),
    loc="upper left",
    borderaxespad=0.5,
    fontsize=12,
    frameon=False  # no box around the legend
)

# -----------------------------
# 9. Tighten layout & save
# -----------------------------
plt.tight_layout()
plt.savefig("/kaggle/working/secondary_structure_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
#secondary structure counts over time (smoothed)
import re
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# File containing DSSP secondary structure counts (xvg format)
filename = "/kaggle/working/count.xvg"  # Change this to your actual file path if different
  # Update this path if necessary

# Smoothing parameters
window_size = 10  # Number of frames for moving average. Adjust as needed.
min_periods = 1   # Minimum number of observations in window to compute average

try:
    # Parse legend labels from header lines starting with "@ s"
    labels = []
    header_lines = []
    with open(filename) as f:
        for line in f:
            if line.startswith("@ s"):
                # Example: @ s0 legend "Loops"
                m = re.match(r'@ s\d+ legend "(.*)"', line)
                if m:
                    labels.append(m.group(1))
                header_lines.append(line)
            elif line.startswith('@') or line.startswith('#'):
                header_lines.append(line)
            else:
                # First non-header line: numeric data begins
                break

    # Read only numeric data lines by filtering out header/comment lines
    numeric_lines = []
    with open(filename) as f:
        for line in f:
            if line.startswith('@') or line.startswith('#'):
                continue
            numeric_lines.append(line)
    # Use StringIO to read filtered lines
    data_str = "".join(numeric_lines)
    df = pd.read_csv(io.StringIO(data_str), sep=r'\s+', header=None)

    # First column is time, the rest are counts
    time = df.iloc[:, 0].values
    data = df.iloc[:, 1:].values

    # Verify labels count matches data columns; if mismatch, warn
    if len(labels) != data.shape[1]:
        print(f"Warning: Found {len(labels)} labels but {data.shape[1]} data columns. Check file format.")

    # Create DataFrame for convenient smoothing
    df_counts = pd.DataFrame(data, columns=labels[:data.shape[1]])
    df_counts.insert(0, "Time", time)

    # Apply moving average smoothing
    df_smoothed = df_counts.copy()
    for col in labels[:data.shape[1]]:
        df_smoothed[col] = df_counts[col].rolling(window=window_size, center=True, min_periods=min_periods).mean()

except FileNotFoundError:

    df_counts = pd.DataFrame(series, columns=labels)
    df_counts.insert(0, "Time", time)
    df_smoothed = df_counts.copy()
    for col in labels:
        df_smoothed[col] = df_counts[col].rolling(window=window_size, center=True, min_periods=min_periods).mean()

# Define colors mapping (user can adjust hex codes to match Grace or their preference)
colors = [
    "#000000",  # Loops
    "#FF0000",  # Breaks
    "#00FF00",  # Bends
    "#0000FF",  # Turns
    "#FF00FF",  # PP_Helices
    "#00FFFF",  # π-Helices
    "#FFFF00",  # 3₁₀-Helices
    "#008000",  # β-Strands
    "#800000",  # β-Bridges
    "#FFA500",  # α-Helices
]
# If more labels than colors, fallback to default cycle
while len(colors) < len(labels):
    colors.append(None)

# Plotting
plt.figure(figsize=(10, 6))
for i, label in enumerate(labels[:df_counts.shape[1]]):
    color = colors[i] if colors[i] else None
    # Plot raw series with low alpha
    # Plot smoothed series with stronger line
    plt.plot(df_smoothed["Time"], df_smoothed[label], label=label, color=color, linewidth=2)

plt.xlabel("Time (ps)")
plt.ylabel("Number of Residues")
plt.title("Smoothed Secondary Structure Counts Over Time")
plt.legend(loc="best", bbox_to_anchor=(1.05, 1), borderaxespad=0.)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
# Optional: save figure
plt.savefig("/kaggle/working/ss_counts_smoothed.png", dpi=300)

plt.show()
